<a href="https://colab.research.google.com/github/SanaAfia/API-Agent/blob/main/DataAnalyticAngentVIT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# INSTALL REQUIRED LIBRARIES
!pip install -q -U google-genai
    #!pip -q install -U pandas
    #!pip -q install -U matplotlib
    #!pip -q install -U python-dotenv


In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from google import genai
from google.genai import types
from google.colab import files
print("Libraries imported successfully!")


Libraries imported successfully!


In [ ]:
from getpass import getpass
api_key = getpass("Paste your Gemini API key: ")

client = genai.Client(api_key=api_key)
print("Gemini client created successfully!")


Paste your Gemini API key: ··········
Gemini client created successfully!


In [ ]:
response = client.interactions.create(
        model="gemini-3.5-flash",
        input="Explain AI agents in one simple sentence."
    )
print(response.output_text)


An AI agent is a smart software program that can independently make decisions and take action to achieve a specific goal.


In [ ]:
    print("Upload your CSV file:")
    uploaded = files.upload()

    # Get the uploaded filename
    filename = list(uploaded.keys())[0]
    print(f"\nUploaded file: {filename}")

    # Load CSV into pandas
    df = pd.read_csv(filename)
    print("\nDataset loaded successfully!")
    print(f"Rows    : {len(df)}")
    print(f"Columns : {len(df.columns)}")
    print("\nColumns:")
    print(list(df.columns))


Upload your CSV file:


Saving amazon.csv to amazon.csv

Uploaded file: amazon.csv

Dataset loaded successfully!
Rows    : 1465
Columns : 16

Columns:
['product_id', 'product_name', 'category', 'discounted_price', 'actual_price', 'discount_percentage', 'rating', 'rating_count', 'about_product', 'user_id', 'user_name', 'review_id', 'review_title', 'review_content', 'img_link', 'product_link']


In [ ]:
def inspect_dataset():
        """
        Gives the agent basic information about the dataset.
        This tool does NOT answer analytical questions.
        It tells the agent:
        - number of rows
        - columns
        - data types
        - missing values
        - sample records
        """
        result = {
            "rows": len(df),
            "columns": list(df.columns),
            "data_types": {
                column: str(dtype)
                for column, dtype in df.dtypes.items()
            },
            "missing_values": {
                column: int(value)
                for column, value in df.isna().sum().items()
            },
            "sample_data": df.head(5).to_dict(orient="records")
        }
        return result


In [ ]:
def analyze_data(pandas_expression: str):
        """
        Analyze the dataset using ONE valid pandas expression.

        IMPORTANT:
        - The pandas_expression MUST be a single Python expression.
        - Do NOT use assignments (=).
        - Do NOT use multiple statements.
        - Do NOT use newline-separated commands.
        - Use pandas method chaining for multiple transformations.
        """
        try:
            result = eval(
                pandas_expression,
                {"df": df, "pd": pd}
            )

            # If result is a pandas Series
            if isinstance(result, pd.Series):
                result = {
                    str(k): str(v)
                    for k, v in result.to_dict().items()
                }
            # If result is a pandas DataFrame
            elif isinstance(result, pd.DataFrame):
                result = result.head(50).to_dict(orient="records")
            # Convert numpy values / other objects into strings
            else:
                try:
                    result = result.item()
                except:
                    result = str(result)

            return {"success": True, "result": result}

        except Exception as e:
            return {"success": False, "error": str(e)}


In [ ]:
def create_chart(chart_type: str, x_column: str, y_column: str, title: str):
        """
        Creates a simple chart from the dataframe.
        Supported chart types: bar, line, scatter
        """
        try:
            plt.figure(figsize=(10, 5))

            if chart_type == "bar":
                data = df.groupby(x_column)[y_column].sum()
                data = data.sort_values(ascending=False).head(10)
                plt.bar(data.index.astype(str), data.values)

            elif chart_type == "line":
                plt.plot(df[x_column], df[y_column], marker="o")

            elif chart_type == "scatter":
                plt.scatter(df[x_column], df[y_column])

            else:
                return {"success": False, "error": "Unsupported chart type."}

            plt.title(title)
            plt.xlabel(x_column)
            plt.ylabel(y_column)
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.show()

            return {"success": True, "message": f"{chart_type} chart created successfully."}

        except Exception as e:
            return {"success": False, "error": str(e)}


In [ ]:
inspect_tool = types.Tool(
        function_declarations=[
            types.FunctionDeclaration(
                name="inspect_dataset",
                description="""
                Inspect the uploaded dataset.
                Use this when you need to understand:
                - number of rows
                - column names
                - data types
                - missing values
                - sample data
                """,
                parameters=types.Schema(
                    type="OBJECT",
                    properties={}
                )
            )
        ]
    )
analyze_tool = types.Tool(
        function_declarations=[
            types.FunctionDeclaration(
                name="analyze_data",
                description="""
                Analyze the dataset to answer questions involving
                totals, averages, grouping, sorting, filtering,
                correlations, or other numerical analysis.
                """,
                parameters=types.Schema(
                    type="OBJECT",
                    properties={
                        "pandas_expression": types.Schema(
                            type="STRING",
                            description="""
                             A valid pandas expression to analyze the dataset.
                             Examples:
                             df["rating"].mean()
                             df["discounted_price"].max()
                             df.groupby("product_name")["discounted_price"].sum()
                             df["rating"].corr(df["rating_count"])
                             """
                        ),
                        "column": types.Schema(
                            type="STRING",
                            description="The numerical column to analyze."
                        ),
                        "group_by": types.Schema(
                            type="STRING",
                            description="Column to group results by, if needed."
                        )
                    },
                    required=["pandas_expression"]
                )
            )
        ]
    )

chart_tool = types.Tool(
        function_declarations=[
            types.FunctionDeclaration(
                name="create_chart",
                description="""
                Create a visualization from the dataset.
                Use this when the user asks for a chart,
                graph, plot, or visualization.
                """,
                parameters=types.Schema(
                    type="OBJECT",
                    properties={
                        "chart_type": types.Schema(
                            type="STRING",
                            description="Type of chart: bar, line, or scatter."
                        ),
                        "x_column": types.Schema(
                            type="STRING",
                            description="Column to use on the X axis."
                        ),
                        "y_column": types.Schema(
                            type="STRING",
                            description="Column to use on the Y axis."
                        ),
                        "title": types.Schema(
                            type="STRING",
                            description="Title of the chart."
                        )
                    },
                    required=["chart_type", "x_column", "y_column"]
                )
            )
        ]
    )

    # Put all our tools together
tools = [inspect_tool, analyze_tool, chart_tool]



In [ ]:
SYSTEM_INSTRUCTION = """
    You are an AI Data Analyst Agent.

    Your job is to answer questions about a CSV dataset.

    You have access to three tools:
    1. inspect_dataset
       - Understand the structure of the dataset.
    2. analyze_data
       - Perform calculations and analysis.
    3. create_chart
       - Create visualizations.

    IMPORTANT:
    - Do not guess numerical answers.
    - Use the appropriate tool whenever you need information from the dataset.
    - You can use multiple tools if necessary.
    - When using analyze_data, provide exactly ONE valid pandas expression.
    - Do not use assignments or multiple statements.
    - Before answering, verify that the requested column actually exists.
    - If the dataset does not contain the requested information, clearly tell the user.
    - Do not silently treat one column as another.
    - After getting the tool results, explain the answer clearly to the user.
    """

config = types.GenerateContentConfig(
        system_instruction=SYSTEM_INSTRUCTION,
        tools=tools
    )
print("Tools configured successfully!")


Tools configured successfully!


In [ ]:
def run_agent(user_question):
        """
        Runs our AI Data Analyst Agent.
        Loop: User question -> Ask LLM -> Tool call? -> Execute tool ->
        Feed result back -> Repeat until final answer.
        """
        print("\n" + "=" * 70)
        print(" AGENT STARTED")
        print("=" * 70)

        # Conversation history
        contents = [
            types.Content(
                role="user",
                parts=[types.Part.from_text(text=user_question)]
            )
        ]

        # available_tools tells Python how to execute those tools
        available_tools = {
            "inspect_dataset": inspect_dataset,
            "analyze_data": analyze_data,
            "create_chart": create_chart
        }

        # Maximum number of tool calls (prevents infinite loops)
        MAX_STEPS = 6
        for step in range(MAX_STEPS):
            print(f"\nAgent step {step + 1}")

            # Ask Gemini what to do
            response = client.models.generate_content(
                model="gemini-3.5-flash-lite",
                contents=contents,
                config=config
            )

            # CHECK WHETHER GEMINI WANTS TO USE A TOOL
            if not response.function_calls:
                print("\n" + "=" * 70)
                print("FINAL ANSWER")
                print("=" * 70)
                print(response.text)
                return response.text

            # Look for function calls
            function_calls = []
            for candidate in response.candidates:
                for part in candidate.content.parts:
                    if part.function_call:
                        function_calls.append(part.function_call)

            # If there are NO tool calls, the agent has its final answer.
            if not function_calls:
                print("\n" + "=" * 70)
                print("FINAL ANSWER")
                print("=" * 70)
                print(response.text)
                return response.text

            # Add the model's response to conversation history
            contents.append(response.candidates[0].content)

            # Execute each requested tool
            for function_call in function_calls:
                tool_name = function_call.name
                tool_args = dict(function_call.args)
                print(f"\nTool selected: {tool_name}")
                print(f"Arguments: {tool_args}")

                # Find the corresponding Python function
                if tool_name not in available_tools:
                    tool_result = {"error": f"Unknown tool: {tool_name}"}
                else:
                    try:
                        function = available_tools[tool_name]
                        tool_result = function(**tool_args)
                    except Exception as e:
                        tool_result = {"error": str(e)}

                print(f"Tool result: {tool_result}")

                # Send the tool result back to Gemini
                contents.append(
                    types.Content(
                        role="user",
                        parts=[
                            types.Part.from_function_response(
                                name=tool_name,
                                response=tool_result
                            )
                        ]
                    )
                )

        # If we reach here, the agent used too many steps.
        print("\nAgent reached maximum number of steps.")
        return "I couldn't complete the analysis within the allowed steps."


In [ ]:
print("\n" + "=" * 70)
print("AI DATA ANALYST AGENT")
print("=" * 70)
print("\nYour dataset:")
print(filename)
print("\nYou can ask questions such as:")
print("""
    • What columns are present?
    • How many rows are there?
    • Which product generated the most revenue?
    • Which product has most discount?
    • What is the average quantity?
    • Show me the top 5 products.
    • Is quantity correlated with revenue?
    • Create a chart of revenue by product.
    • Show me a trend in the data.
    """)
print("\nType 'exit' to stop.")

while True:
        question = input("\n You: ").strip()
        if question.lower() in ["exit", "quit", "q"]:
            print("\n Goodbye!")
            break
        if not question:
            continue
        run_agent(question)



AI DATA ANALYST AGENT

Your dataset:
amazon.csv

You can ask questions such as:
    
    • What columns are present?    
    • How many rows are there?    
    • Which product generated the most revenue?    
    • Which product has most discount?    
    • What is the average quantity?    
    • Show me the top 5 products.    
    • Is quantity correlated with revenue?    
    • Create a chart of revenue by product.    
    • Show me a trend in the data.    
    

Type 'exit' to stop.

 AGENT STARTED

Agent step 1

FINAL ANSWER
Hi Sana! How can I help you analyze your dataset today? Please let me know what dataset you are working with or what questions you have.

 AGENT STARTED

Agent step 1

FINAL ANSWER
Good luck with your hackathon! Let me know if you need to inspect a dataset, analyze some data, run calculations, or create charts for your project. I'm ready to help!
